# 🎯 Module 12: Recommendation Systems

> **Stage:** Machine Learning  
> **Level:** Intermediate  
> **Module:** 12 — Recommendation Systems  
> **Date:** 24 January 2026  
> **Previous:** 11 — Time Series  
> **Next:** 13 — NLP Basics

---

**Recommendation systems** predict what a user may want based on **user**, **item**, and **interaction** data.

Think Netflix, YouTube, Amazon, Spotify.

In this notebook we'll cover:

- **Content-based filtering**
- **Collaborative filtering** (user-based, item-based)
- **User-item interaction matrix**
- **Matrix factorization** (latent factors)
- **SVD** as a classical decomposition
- **Hybrid systems**
- **Cold start** problem
- Candidate generation + ranking workflow

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain the **three families** of recommenders
- Build a **user-item interaction matrix**
- Implement **content-based** filtering with TF-IDF + cosine similarity
- Implement **user-based** and **item-based** collaborative filtering
- Use **matrix factorization** (SVD) to predict preferences
- Explain the **cold-start** problem and mitigation strategies
- Distinguish **explicit vs implicit** feedback
- Understand **candidate generation + ranking** in production systems

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

np.random.seed(42)
pd.set_option('display.max_columns', None)
print('✅ Setup ready.')

---
## 🧠 1. What Is a Recommendation System?

A recommendation system predicts what a user may want, based on:

- The **user**
- The **items**
- The **interactions** between them

### Basic idea

```text
User + Items + Past Behavior
            ↓
   Recommendation Model
            ↓
   Ranked Recommendations
```

> The goal is often **ranking relevant items** for each user, not just predicting a rating.

---
## 🗂️ 2. Three Main Approaches

```text
Recommendation Systems
├── Content-Based Filtering
├── Collaborative Filtering
└── Hybrid Systems
```

Each has its own strengths and weaknesses — we'll explore all three.

---
## 🎬 3. Content-Based Filtering

**Content-based filtering** recommends items **similar to items the user already likes**.

It uses **item features** — genre, director, keywords, description, etc.

### Workflow

```text
User's Previous Likes
        ↓
Create User Profile
        ↓
Compare with Item Features
        ↓
Calculate Similarity
        ↓
Rank Items
```

### Cosine Similarity

$$\cos(\theta) = \frac{A \cdot B}{\|A\| \, \|B\|}$$

Values close to `1` indicate **greater directional similarity**.

### ✅ Advantage
Works well with **few users** — relies on item content, not user history.

### ⚠️ Limitation
Tends to keep recommending **the same kinds of items** — hard to introduce new tastes.

In [ ]:
# Build a small movie catalog with text features
movies = pd.DataFrame([
    {'title': 'Space Heroes',    'genres': 'action sci-fi superhero space'},
    {'title': 'Galaxy War',      'genres': 'action sci-fi space war'},
    {'title': 'City Detective',  'genres': 'crime thriller city detective'},
    {'title': 'Romance in Paris','genres': 'romance drama paris love'},
    {'title': 'Laughing Out',    'genres': 'comedy funny family'},
    {'title': 'Iron Warriors',   'genres': 'action sci-fi superhero war'},
    {'title': 'Silent Mystery',  'genres': 'mystery crime thriller detective'},
    {'title': 'First Love',      'genres': 'romance drama love'},
])

print(movies)

In [ ]:
# TF-IDF on genres + cosine similarity → content-based recommendations
tfidf = TfidfVectorizer()
item_matrix = tfidf.fit_transform(movies['genres'])

sim_matrix = cosine_similarity(item_matrix)

def recommend_content(title, sim_matrix, top_n=3):
    idx = movies.index[movies['title'] == title][0]
    scores = list(enumerate(sim_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [s for s in scores if s[0] != idx][:top_n]
    return [(movies.iloc[i]['title'], round(score, 3)) for i, score in scores]

print('Because you liked "Space Heroes", we recommend:')
for title, score in recommend_content('Space Heroes', sim_matrix):
    print(f'  {title:<20} (similarity: {score})')

---
## 👥 4. Collaborative Filtering

Collaborative filtering uses the **behavior of many users**.

Instead of asking:

> "What is this item about?"

It asks:

> "What did **similar users** like?"

### Example

```text
User A → Movie 1 ✓
User A → Movie 2 ✓
User B → Movie 1 ✓
User B → Movie 2 ?     ← recommend Movie 2
```

Because User A and User B have similar preferences.

---
## 📊 5. User-Item Interaction Matrix

The core data structure for collaborative filtering:

| | Movie A | Movie B | Movie C |
| --- | ---: | ---: | ---: |
| User 1 | 5 | 4 | ? |
| User 2 | 5 | ? | 4 |
| User 3 | 1 | 2 | 5 |

`?` = unknown interaction.

### Interactions aren't always ratings

```text
Click
View
Like
Purchase
Watch time
Add to cart
```

In [ ]:
# Build a small user-item rating matrix
users  = ['U1', 'U2', 'U3', 'U4', 'U5']
items  = ['Item A', 'Item B', 'Item C', 'Item D', 'Item E']

R = pd.DataFrame([
    [5, 4, np.nan, 1, 2],
    [4, 5, np.nan, np.nan, 3],
    [np.nan, 3, 4, 4, np.nan],
    [2, np.nan, 5, 5, 4],
    [1, 2, 4, 5, 5],
], index=users, columns=items)

print('User-Item matrix:')
print(R)
print(f'\nSparsity: {R.isna().sum().sum() / R.size:.1%} missing')

---
## 🔄 6. User-Based vs Item-Based

### User-Based Collaborative Filtering

Find **users with similar behavior** and recommend what they liked:

```text
User A
  ↓
Similar Users
  ↓
Items they liked
  ↓
Recommendations
```

### Item-Based Collaborative Filtering

Find **items commonly interacted with together**:

```text
Item A
  ↓
Similar Items
  ↓
Recommend Similar Items
```

> In large systems, **item-based** is often easier to maintain — item relationships are more **stable** than user-user relationships.

In [ ]:
# User-user similarity
R_filled = R.fillna(0)   # for similarity computation
user_sim  = cosine_similarity(R_filled)
user_sim_df = pd.DataFrame(user_sim, index=users, columns=users)
print('User-user similarity:')
print(user_sim_df.round(3))

# Item-item similarity
item_sim  = cosine_similarity(R_filled.T)
item_sim_df = pd.DataFrame(item_sim, index=items, columns=items)
print('\nItem-item similarity:')
print(item_sim_df.round(3))

In [ ]:
# Item-based recommendation: given an item, find similar items
def similar_items(item_name, item_sim_df, top_n=3):
    scores = item_sim_df[item_name].sort_values(ascending=False)
    scores = scores.drop(item_name).head(top_n)
    return scores

print('Because you liked "Item A", you might also like:')
print(similar_items('Item A', item_sim_df).round(3))

In [ ]:
# User-based recommendation: predict missing rating for a target user
def predict_user_based(user, item, R, user_sim_df, k=2):
    if item in R.loc[user] and not np.isnan(R.loc[user, item]):
        return R.loc[user, item]   # already known
    sims = user_sim_df[user].drop(user)
    rated = R[item].dropna()
    common = sims.index.intersection(rated.index)
    if len(common) == 0:
        return np.nan
    top_k = sims.loc[common].sort_values(ascending=False).head(k)
    num = sum(sims[u] * R.loc[u, item] for u in top_k.index)
    den = sum(abs(sims[u]) for u in top_k.index) + 1e-9
    return num / den

for u, it in [('U1', 'Item C'), ('U2', 'Item D'), ('U3', 'Item A')]:
    pred = predict_user_based(u, it, R, user_sim_df, k=2)
    print(f'Predicted rating for {u} on {it}: {pred:.3f}')

---
## 🧬 7. Matrix Factorization

A user-item matrix is usually **sparse**. **Matrix factorization** represents users and items with lower-dimensional latent vectors:

$$R \approx U \times V^T$$

| Symbol | Meaning |
| --- | --- |
| `R` | user-item interaction matrix |
| `U` | user latent factors |
| `V` | item latent factors |

### Latent factors might represent hidden concepts

```text
Action preference
Comedy preference
Price sensitivity
Genre preference
```

The model **learns** these from data — we don't hand-label them.

In [ ]:
# Matrix factorization with Truncated SVD
R_filled = R.fillna(0)

svd = TruncatedSVD(n_components=2, random_state=42)
U = svd.fit_transform(R_filled)             # user latent factors
V = svd.components_.T                        # item latent factors

R_pred = U @ V.T                             # reconstructed matrix

print('User latent factors (2D):')
print(pd.DataFrame(U, index=users).round(3))
print('\nItem latent factors (2D):')
print(pd.DataFrame(V, index=items).round(3))
print('\nReconstructed ratings:')
print(pd.DataFrame(R_pred, index=users, columns=items).round(2))

In [ ]:
# Recommend items with the highest predicted score for each user
R_pred_df = pd.DataFrame(R_pred, index=users, columns=items)

print('Top-2 recommendations per user (from SVD):')
for u in users:
    # exclude items the user already rated (if they rated them)
    rated_items = R.loc[u].dropna().index
    candidates = R_pred_df.loc[u].drop(rated_items, errors='ignore')
    top2 = candidates.sort_values(ascending=False).head(2)
    print(f'  {u}: ' + ', '.join(f'{it} ({sc:.2f})' for it, sc in top2.items()))

---
## 🧩 8. Hybrid Recommendation Systems

A **hybrid system** combines multiple approaches:

```text
             ┌── Content-Based ──┐
User Data ───┤                   ├──→ Combined Score
             └ Collaborative ────┘
                         ↓
                   Recommendations
```

### E-commerce example

```text
Product similarity
        +
User purchase history
        +
Similar users
        +
Current session behavior
        ↓
Final Ranking
```

> Hybrid systems help cover the weaknesses of any single approach.

In [ ]:
# Simple hybrid: weighted mix of content-similarity and collaborative-similarity
def hybrid_recommend(user, seed_item, R_pred_df, item_sim_df, alpha=0.5, top_n=3):
    # Collaborative score: predicted rating for user on every item
    collab = R_pred_df.loc[user]
    # Content score: similarity of every item to the seed item the user liked
    content = item_sim_df[seed_item]
    # Combine
    combined = alpha * content + (1 - alpha) * collab
    # Exclude seed
    combined = combined.drop(seed_item, errors='ignore')
    return combined.sort_values(ascending=False).head(top_n)

print('Hybrid recommendations for U1 (seed = Item A, alpha=0.5):')
print(hybrid_recommend('U1', 'Item A', R_pred_df, item_sim_df).round(3))

---
## ❄️ 9. The Cold-Start Problem

### New User

```text
New User
   ↓
No history
   ↓
What should we recommend?
```

**Possible solutions:**

- Ask for preferences
- Recommend popular items
- Use demographic/context info where appropriate
- Use content-based recommendations
- Learn from first interactions

### New Item

A newly added item has **no interaction history** — content-based features can help surface it before collaborative signals exist.

In [ ]:
# Simulate cold start: add a new user with no ratings
R_cold = R.copy()
R_cold.loc['U_new'] = np.nan

print('New user has no ratings.')
print('\nFallback: recommend globally popular items (average rating):')
pop = R.mean().sort_values(ascending=False)
print(pop.round(2))

print('\nAfter first click on a seed item, switch to content-based:')
seed = pop.index[0]
print(f'  Seed item: {seed}')
print('  Recommendations from content similarity:')
print(similar_items(seed, item_sim_df).round(3))

---
## 📝 10. Explicit vs Implicit Feedback

### Explicit
User **directly** tells you their preference:

```text
Ratings (1–5 stars)
Likes / Dislikes
Thumbs up / down
```

### Implicit
You **infer** preference from behavior:

```text
Clicks
Views
Watch time
Purchases
Add to cart
```

Most **production systems** rely heavily on **implicit** signals because far more data is available.

---
## 🚀 11. Production Workflow

```text
Collect User + Item Data
          ↓
Create Interaction Matrix
          ↓
Clean / Transform Data
          ↓
Choose Approach
          ↓
Content / Collaborative / Hybrid
          ↓
Generate Candidate Items
          ↓
Calculate Scores
          ↓
Rank Recommendations
          ↓
Evaluate
          ↓
Deploy
```

> Modern systems use **candidate generation** (fast, recall-focused) + **ranking** (slow, precision-focused).

---
## 🧪 12. Practice Checklist

- [ ] Build a movie recommendation dataset
- [ ] Create a user-item interaction matrix
- [ ] Implement content-based recommendations with TF-IDF + cosine similarity
- [ ] Build a simple collaborative filtering system
- [ ] Experiment with matrix factorization
- [ ] Recommend similar products from purchase history
- [ ] Create a simple hybrid recommender
- [ ] Demonstrate the cold-start problem
- [ ] Rank the top-5 recommendations for a user

---
## 🏋️ 13. Hands-On Exercises

### Exercise 1 — TF-IDF Parameters
Try `TfidfVectorizer(stop_words='english', ngram_range=(1,2))`. Do recommendations improve?

### Exercise 2 — Ratings vs Binary
Convert the rating matrix into **binary** (liked / not-liked) and repeat the collaborative filtering. Which is more stable?

### Exercise 3 — SVD n_components Sweep
Try `n_components ∈ {2, 3, 5, 10}`. How does reconstruction error change? What is the effect on top-k recommendations?

### Exercise 4 — Hybrid Alpha Sweep
Try `alpha ∈ {0.1, 0.3, 0.5, 0.7, 0.9}`. When is **content** more important? When is **collaborative** more important?

### Exercise 5 — Popularity Baseline
Add a **popularity baseline**: recommend the most-rated items to everyone. Do the collaborative recommenders beat it?

### Exercise 6 — Implicit Feedback
Replace explicit ratings with click counts. Repeat the pipeline. Which results change?

### Exercise 7 — Cold-Start Strategies
Simulate a new user with 1 rating. Compare (a) popularity fallback, (b) content-based, (c) hybrid. Which wins on the next 5 interactions?

---
## 🎤 14. Interview Questions

1. **What is a recommendation system?**  
   A model that predicts what items a user may want, based on user, item, and interaction data.

2. **Content-based vs collaborative filtering?**  
   Content-based uses item features. Collaborative uses patterns across users and items.

3. **What is a user-item interaction matrix?**  
   A matrix with users as rows, items as columns, and interactions (ratings, clicks) as values.

4. **User-based vs item-based collaborative filtering?**  
   User-based finds similar users. Item-based finds similar items.

5. **What is matrix factorization?**  
   Decomposing the user-item matrix into low-dimensional user and item latent factors.

6. **What is SVD?**  
   Singular Value Decomposition — a classical matrix factorization that reveals latent structure.

7. **What is cosine similarity?**  
   A measure of the angle between two vectors: $\cos\theta = A \cdot B / (\|A\|\|B\|)$.

8. **What is the cold-start problem?**  
   Difficulty recommending for new users or new items with little/no interaction history.

9. **How can you recommend a new item with no interaction history?**  
   Use its content features (content-based filtering) until collaborative signals accumulate.

10. **Why use a hybrid recommender?**  
    To combine complementary strengths and reduce the weaknesses of any single approach.

11. **Explicit vs implicit feedback?**  
    Explicit = ratings, likes. Implicit = clicks, views, purchases.

12. **Why is ranking important in recommendation systems?**  
    Users see a ranked list — order matters more than raw scores.

---
## 🏁 15. Key Takeaways

- **Content-Based Filtering** → recommends items similar to what the user already likes
- **Collaborative Filtering** → uses patterns across users and items
  - **User-Based** → similar users
  - **Item-Based** → similar items
- **Matrix Factorization** → learns latent user/item representations
- **SVD** → classical latent-factor decomposition
- **Hybrid Systems** → combine multiple approaches
- **Cosine Similarity** → common vector similarity metric
- **Cold Start** → new users / new items with no history
- Production systems **generate candidates + rank**

### 🧭 Golden Rule

> **No single recommender is best. Match the approach to your data: content features, user interactions, or both — and always handle cold start explicitly.**

---

### 🔗 What's Next?

**Module 13 — NLP Basics**

We shift to **text** — text preprocessing, **Bag of Words**, **TF-IDF**, **word embeddings**, and **sentiment analysis**.